# Planted circuits and population trajectories

Start here before fitting connectivity estimators. Four controls separate a favorable one-to-one correspondence, distinct circuits with exactly matched stationary population marginals, a fixed circuit with different initial histories, and a stationary process without switching or resets. These use directly observed linear Gaussian activity. Truth is stored only in recording metadata, never supplied as behavior or predictor features. Notebook 04 adds repeated independent seeds and declared observation stresses.


In [ ]:
import logging
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]

if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)

ROOT = _REPOSITORY_ROOT
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logging.getLogger("effectome").setLevel(logging.INFO)
_CONVENTIONAL_PYTHON = ROOT / "tools/conventional-cgc/.venv/bin/python"
print("Conventional CPU runtime:", _CONVENTIONAL_PYTHON)
print("Runtime installed:", _CONVENTIONAL_PYTHON.is_file())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from effectome.experiments.transition_examples import (
    TRANSITION_EXAMPLE_SCENARIOS,
    TransitionExampleConfig,
    make_transition_example,
)

print("Project:", ROOT)
print("Python:", sys.version.split()[0])


## Generate all four controls

Change the seed, dwell, contraction, neuron count, or recording length to explore sensitivity. A dwell shorter than the connectivity history deliberately mixes circuits inside a fitted window. Use longer recordings for inference; these small examples are engineering and interpretation checks.


In [ ]:
SEED = 42
examples = {
    scenario: make_transition_example(TransitionExampleConfig(scenario=scenario, seed=SEED))
    for scenario in TRANSITION_EXAMPLE_SCENARIOS
}
display(pd.DataFrame([
    {
        "scenario": scenario,
        "samples": recording.n_timepoints,
        "true_circuits": len(recording.metadata["true_coefficients"]),
        "independent_trials": len(recording.metadata["valid_ranges"]),
        "matched_stationary_marginals": recording.metadata["identical_stationary_marginals"],
    }
    for scenario, recording in examples.items()
]))


## Observed activity and the generating circuits

The first panel uses the first two observed neurons, not a fitted manifold. Colors denote planted circuit identity except for the fixed-circuit example, where colors denote initial-condition classes. The matrices include autoregressive self-history on their diagonals. Later exported effectome graphs may suppress that diagonal; the full generating dynamics still retain it.


In [ ]:
fig, axes = plt.subplots(len(examples), 4, figsize=(17, 11), constrained_layout=True)
for row, (scenario, recording) in enumerate(examples.items()):
    truth = recording.metadata
    matrices = truth["true_coefficients"]
    colors = truth["true_initial_condition_class"] if scenario == "fixed_circuit" else truth["true_regime"]
    axes[row, 0].scatter(*recording.traces[:2], c=colors, s=4, alpha=0.5, cmap="coolwarm")
    axes[row, 0].set(xlabel="Observed neuron 0", ylabel="Observed neuron 1", title=scenario)
    scale = np.max(np.abs(matrices))
    for column in (1, 2):
        index = min(column - 1, len(matrices) - 1)
        image = axes[row, column].imshow(matrices[index], cmap="coolwarm", vmin=-scale, vmax=scale)
        axes[row, column].set(
            title=f"True W[{index}]" + (" (same circuit)" if len(matrices) == 1 else ""),
            xlabel="Target neuron", ylabel="Source neuron",
        )
        fig.colorbar(image, ax=axes[row, column], shrink=0.7)
    stop = min(recording.n_timepoints, 600)
    axes[row, 3].plot(recording.time[:stop], recording.traces[0, :stop], linewidth=0.8)
    axes[row, 3].set(xlabel="Time (s)", ylabel="Neuron 0 activity", title="First 600 samples")
plt.show()


## Why the matched-marginal construction works

For row-vector dynamics x(t+1) = x(t) W + epsilon, stationary covariance obeys Sigma = W.T Sigma W + Q. With W = r R for either orthogonal rotation R, Sigma = I and Q = (1-r²) I satisfy this identity. Starting with a Gaussian N(0,I) draw preserves the same Gaussian marginal under arbitrary deterministic switching. The conditional dynamics and lagged correlations differ. This establishes equality of the planted stationary marginal, not equality of finite-sample estimates or of all time-window statistics.


In [ ]:
matched = examples["matched_marginals"]
truth = matched.metadata
for index, (matrix, noise, covariance) in enumerate(zip(
    truth["true_coefficients"],
    truth["true_innovation_covariances"],
    truth["true_stationary_covariances"],
    strict=True,
)):
    residual = matrix.T @ covariance @ matrix + noise - covariance
    print(f"Circuit {index}: stationary-covariance residual = {np.max(np.abs(residual)):.3g}")
    print("Stationary mean:", truth["true_stationary_means"][index])
    print("Stationary covariance diagonal:", np.diag(covariance))
print("Distinct coefficients:", not np.allclose(*truth["true_coefficients"]))


## One fixed circuit, two initial conditions

The trajectories below share the exact same W. Each trial starts at +4 or -4 on neuron 0; resets are declared temporal boundaries. Different trajectories may be fully explained by observed position and history, so this control should not be labeled hidden circuit switching or biological necessity.


In [ ]:
fixed = examples["fixed_circuit"]
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
for trial, (start, stop) in enumerate(fixed.metadata["valid_ranges"][:2]):
    path = fixed.traces[:2, start:stop]
    ax.plot(*path, alpha=0.8, label=f"Trial {trial}: initial class {trial % 2}")
    ax.scatter(*path[:, 0], s=80, marker="x")
ax.set(
    xlabel="Observed neuron 0", ylabel="Observed neuron 1",
    title="Same circuit, different initial histories",
)
ax.legend()
plt.show()


Continue with 02_shared_transition_comparison.ipynb to fit the common reference, or 03_correspondence_and_controls.ipynb to inspect planted correspondences before estimation.
